In [ ]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload()  # Browse and select all six CSVs
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()
def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")

**SECTION 01 / DATA PREPARATION & ANALYSIS**

In [ ]:
#1
s_master_df=spark.read.csv("substation_master.csv",header=True,inferSchema=True)
t_master_df=spark.read.csv("tariff_master.csv",header=True,inferSchema=True)
maintenance_df=spark.read.csv("maintenance_tickets_1600.csv",header=True,inferSchema=True)
m_master_df=spark.read.csv("meter_master.csv",header=True,inferSchema=True)
m_readings_df=spark.read.csv("meter_readings_90000.csv",header=True,inferSchema=True)
m_extra_df=spark.read.csv("meter_readings_extra_12000.csv",header=True,inferSchema=True)




In [ ]:
#1Load all six CSVs. Display the schema, row count and first 10 rows of the main readings DataFrame.

s_master_df.printSchema()
print(s_master_df.count())
m_readings_df.printSchema()
print(m_readings_df.count())


In [ ]:
#2 Select reading_id, meter_id, reading_timestamp, energy_kwh and reading_status. In a separate DataFrame, rename source_type to collection_mode.
mr_df=m_readings_df.select("reading_id","meter_id","reading_timestamp","energy_kwh","reading_status")
mr_df.show(5)
mr_df=m_readings_df.withColumnRenamed("source_type","collectio_mode")
mr_df.show(3)

In [ ]:
m_readings_df.show(3)

In [ ]:
#3  Filter readings where reading_status is Valid and energy_kwh exceeds 20. Sort by energy consumed, highest first.
from pyspark.sql.functions import col
mr_df=m_readings_df.filter((col("reading_status")=="Valid") & (col("energy_kwh")>20))
mr_df.orderBy(col("energy_kwh").desc()).show(5)
mr_df.show(10)


In [ ]:
#4  Create energy_band using when() / otherwise(): Low below 5 kWh, Medium from 5 through 25 kWh, High above 25 kWh, and Unknown for nulls.
from pyspark.sql.functions import when
mr_df=mr_df.withColumn("energy_band",
                       when(col("energy_kwh").isNull(),"Unknown")
                       .when(col("energy_kwh")<5,"low")
                       .when(col("energy_kwh")<=25,"Medium")
                       .otherwise("High")
                       )
mr_df.show(5)

In [ ]:
# 5. Calculate null counts for energy_kwh, voltage_v, power_factor and operator_id.
print(
    m_readings_df.filter(col("energy_kwh").isNull()).count(),
    m_readings_df.filter(col("voltage_v").isNull()).count(),
    m_readings_df.filter(col("power_factor").isNull()).count(),
    m_readings_df.filter(col("operator_id").isNull()).count()
)


In [ ]:
#6 Create a new DataFrame that fills missing operator_id with UNASSIGNED and missing power_factor with 0.90.
mr_df=mr_df.na.fill({"operator_id":"UNASSIGNED"})
mr_df=mr_df.na.fill({"power_factor":0.90})
mr_df.select("operator_id","power_factor").show()

In [ ]:
#7 . Remove readings with missing energy_kwh or voltage_v. Report the number of rows removed
null_mr_df=mr_df.filter((col("energy_kwh").isNull()) | (col("voltage_v").isNull()))
print(null_mr_df.count())
cleaned_df = mr_df.filter(
    col("energy_kwh").isNotNull() & col("voltage_v").isNotNull()
)

In [ ]:
#8  Join readings with meter_master on meter_id. Calculate total energy for Valid readings by site_type.
from pyspark.sql.functions import col, sum
join_df=m_readings_df.join(m_master_df,on="meter_id",how="inner")
df=(join_df.filter(col("reading_status")=="Valid")
    .groupBy(col("site_type"))
    .agg(sum("energy_kwh").alias("total_energy_for_valid_readings")))
df.show()

In [ ]:
s_master_df.show(3)

In [ ]:
#9 Extend the join to substation_master. Calculate total Valid energy by region and city.
join_df_sm=join_df.join(s_master_df,on="substation_id",how="inner")
df2=(join_df_sm.filter(col("reading_status")=="Valid")
     .groupBy("region","city")
     .agg(sum("energy_kwh").alias("total_valid_energy")))
df2.show()

In [ ]:
m_master_df.show(2)

In [ ]:
#10 Join tariff_master through meter_master. For non-Rejected readings with non-null energy, calculate energy_charge_inr = energy_kwh * rate_per_kwh_inr. Do not add the monthly fixed charge to every reading.
join_df_tariff_meter = m_readings_df.join(m_master_df, on="meter_id", how="inner")
join_df_tariff_meter = join_df_tariff_meter.join(t_master_df, on="tariff_id", how="inner")
df=join_df_tariff_meter.filter((col("reading_status")!="Rejected") & (col("energy_kwh").isNotNull()))
df.withColumn("energy_charge_inr",col("energy_kwh")*col("rate_per_kwh_inr"))
df.show(5)

In [ ]:
m_master_df.show(1)
maintenance_df.show(1)

In [ ]:
from pyspark.sql.functions import avg,count
#11 . Join maintenance tickets with meter_master. Find ticket counts and average maintenance_cost_inr by site_type and severity.
join_df_main_master=maintenance_df.join(m_master_df,on="meter_id",how="inner")
df=(join_df_main_master.groupBy("site_type","severity")
.agg((count("*").alias("ticket_counts")),(avg("maintenance_cost_inr").alias("Avg_maintenance_cost")))
)
df.show(5)


In [ ]:
#12 Calculate average voltage_v and total outage_minutes by substation. Show the five substations with the highest total outage minutes.
df=(join_df_tariff_meter.groupBy("substation_id")
    .agg((avg("voltage_v").alias("Avg_Voltage")),(sum("outage_minutes").alias("total_outage_minutes")))
)
df.orderBy(col("total_outage_minutes").desc()).show(5)

In [ ]:
#13 . Calculate the number of Valid, Estimated and Rejected readings for each source_type, ordered by reading count

df = (m_readings_df.groupBy("source_type")
      .agg(
          sum(when(col("reading_status") == "Valid", 1).otherwise(0)).alias("Valid_count"),
          sum(when(col("reading_status") == "Estimated", 1).otherwise(0)).alias("Estimated_count"),
          sum(when(col("reading_status") == "Rejected", 1).otherwise(0)).alias("Rejected_count"),
          count("*").alias("reading_count")
      )
      .orderBy(col("reading_count").desc())
)

df.show()

**SECTION 02 / ADVANCED DATAFRAME OPERATIONS**

In [ ]:
#14 Convert reading_timestamp into a Spark timestamp. Extract the year and month and calculate monthly Valid energy consumption by region.
from pyspark.sql.functions import to_date,to_timestamp,month,year
from pyspark.sql.functions import col, to_timestamp, year, month, sum

df = m_readings_df.withColumn(
    "reading_timestamp", to_timestamp(col("reading_timestamp"))
)
df = df.withColumn("year", year(col("reading_timestamp")))
df = df.withColumn("month", month(col("reading_timestamp")))

df = df.join(m_master_df, on="meter_id", how="inner")

df = df.join(s_master_df, on="substation_id", how="inner")

df = (df.filter(col("reading_status") == "Valid")
      .groupBy("region", "year", "month")
      .agg(sum("energy_kwh").alias("monthly_valid_energy"))
)

df.show()


In [ ]:

#15. Find the top three meters by total Valid energy within each substation using dense_rank().
from pyspark.sql.functions import  dense_rank
from pyspark.sql.window import Window


join_df = m_readings_df.join(m_master_df, on="meter_id", how="inner")


df = (join_df
      .filter(col("reading_status") == "Valid")
      .groupBy("substation_id", "meter_id")
      .agg(sum("energy_kwh").alias("total_valid_energy")))


window_spec = Window.partitionBy("substation_id").orderBy(
    col("total_valid_energy").desc()
)

df = df.withColumn("rank", dense_rank().over(window_spec))

df.filter(col("rank") <= 3).show()

In [ ]:
#16 Calculate cumulative Valid energy for each meter using a window ordered by reading timestamp and reading ID.

running_energy_window = Window.partitionBy("meter_id").orderBy("reading_timestamp", "reading_id")

readings_df.filter(F.col("reading_status") == "Valid").withColumn(
    "cumulative_valid_energy_kwh",
    F.sum("energy_kwh").over(running_energy_window)
).select(
    "meter_id",
    "reading_timestamp",
    "reading_id",
    "energy_kwh",
    "cumulative_valid_energy_kwh"
).orderBy("meter_id", "reading_timestamp", "reading_id").show()

In [ ]:
#17  Combine both readings files using unionByName(). Compare row counts before and after the union.
print("Row count before combining(m_readings_df): ",m_readings_df.count())
print("Row count before combining(m_extra_df): ",m_extra_df.count())
combined_df=m_readings_df.unionByName(m_extra_df)
print("Row count after combining: ",combined_df.count())

In [ ]:
#18 Identify repeated reading_id values, remove duplicates using that key, and report the final unique reading count.
df=(combined_df.groupBy(col("reading_id"))
.agg(count("*").alias("total_count"))
.filter(col("total_count")>1))
df.show()
unique_df=df.dropDuplicates(["reading_id"])
unique_df.count()



In [ ]:
#19 Build a pipeline with filter(), withColumn(), groupBy() and orderBy(). Inspect it using explain() before calling an action.
from pyspark.sql.functions import lit


df = (m_readings_df
      .filter(col("reading_status") == "Valid")
      .withColumn("country", lit("India"))
      .groupBy("country", "reading_status")
      .agg(sum("energy_kwh").alias("total_energy"))
      .orderBy(col("total_energy").desc())
)

df.explain()

df.show()

In [ ]:
#20 Cache Valid readings, run count() twice, and record the elapsed time for each action. Check is_cached.
import time
df=m_readings_df.filter(col("reading_status")=="Valid")
start=time.perf_counter()
df.count()
end=time.perf_counter()
print("Execution before cache: " , end-start," seconds")
df.cache()
print(df.is_cached)
start=time.perf_counter()
df.count()
end=time.perf_counter()
print("Execution after cache: " , end-start," seconds")


In [ ]:
#21
from pyspark import StorageLevel
df=m_readings_df.filter(col("outage_minutes")>0)
df.persist(StorageLevel.MEMORY_AND_DISK)
df.count()
df.show()
df.unpersist()

In [ ]:
#22
start=time.perf_counter()
df=m_readings_df.agg(avg("energy_kwh").alias("Avg_energy"))
df.show(5)
end=time.perf_counter()
print("before cache: ",end-start," seconds")

df=m_readings_df.agg(avg("energy_kwh").alias("Avg_energy"))
df.cache()
start=time.perf_counter()
df.show(5)
end=time.perf_counter()
print("After cache:", end-start," seconds")

In [ ]:
#23
from pyspark.sql.functions import spark_partition_id
df = m_readings_df.repartition(8)

df = df.withColumn("partition_id", spark_partition_id())

df.groupBy("partition_id").agg(
    count("*").alias("row_count")
).orderBy("partition_id").show()

In [ ]:
#24
df = m_readings_df.repartition(12)

print("Original partitions:", df.rdd.getNumPartitions())

# Reduce using coalesce()
df1 = df.coalesce(3)

print("Coalesce partitions:", df1.rdd.getNumPartitions())
df1.explain()

# Reduce using repartition()
df2 = df.repartition(3)

print("Repartition partitions:", df2.rdd.getNumPartitions())
df2.explain()

In [ ]:
#25
df = cleaned_df.join(m_master_df, on="meter_id", how="inner")


df = df.join(s_master_df, on="substation_id", how="inner")

df = df.repartition(4)

df.write.mode("overwrite").parquet("/content/energy_output")


parquet_df = spark.read.parquet("/content/energy_output")


parquet_df.show(5)
print("Output partitions:", parquet_df.rdd.getNumPartitions())

**Final Capstone**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
# 1. Combine both readings datasets and deduplicate by reading_id

combined_df = m_readings_df.unionByName(m_extra_df)

print("Rows before deduplication:", combined_df.count())

combined_df = combined_df.dropDuplicates(["reading_id"])

print("Rows after deduplication:", combined_df.count())


In [ ]:
#2. Apply cleaning rule for null energy values Exclude Rejected readings from billable consumption.

cleaned_df = combined_df.withColumn(
    "billable_energy_kwh",
    F.when(
        (F.col("reading_status") != "Rejected") &
        (F.col("energy_kwh").isNotNull()),
        F.col("energy_kwh")
    ).otherwise(0)
)
cleaned_df.show(5)




In [ ]:
# 3. Join meter, substation and tariff information

join_df = cleaned_df.join(
    m_master_df, on="meter_id", how="inner"
)

join_df = join_df.join(
    s_master_df, on="substation_id", how="inner"
)

join_df = join_df.join(
    t_master_df, on="tariff_id", how="inner"
)

In [ ]:

# 4. Create month column and calculate total kWh and estimated energy charges

join_df = join_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp")
)

join_df = join_df.withColumn(
    "month",
    F.date_format("reading_timestamp", "yyyy-MM")
)

join_df = join_df.withColumn(
    "energy_charge_inr",
    F.col("billable_energy_kwh") * F.col("rate_per_kwh_inr")
)
join_df.show(5)

In [ ]:
# 5. Calculate total outage minutes and outage percentage by region and site_type

join_df = join_df.withColumn(
    "outage_reported",
    F.when(F.col("outage_minutes") > 0, 1).otherwise(0)
)

report_df = (join_df
    .groupBy("month", "region", "site_type")
    .agg(
        F.sum("billable_energy_kwh").alias("total_billable_kwh"),
        F.sum("energy_charge_inr").alias("total_energy_charges_inr"),
        F.sum(F.coalesce(F.col("outage_minutes"), F.lit(0))).alias("total_outage_minutes"),
        F.count("*").alias("total_readings"),
        F.sum("outage_reported").alias("outage_readings")
    )
)

report_df = report_df.withColumn(
    "outage_percentage",
    (F.col("outage_readings") / F.col("total_readings")) * 100
)
report_df.show(5)

In [ ]:
# 6. Rank site types within each region by total billable energy charges
rank_window = Window.partitionBy(
    "region", "month"
).orderBy(
    F.col("total_energy_charges_inr").desc()
)

report_df = report_df.withColumn(
    "site_type_rank",
    F.dense_rank().over(rank_window)
)

report_df = report_df.orderBy(
    "region", "month", "site_type_rank"
)

report_df.show()

In [ ]:
# 7. Cache final aggregated report, run two actions and release cached data

report_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Total report rows:", report_df.count())

report_df.show(10)

report_df.unpersist()

In [ ]:
#8. Save final report as Parquet with two partitions


report_df.repartition(2).write.mode("overwrite").parquet(
    "/content/final_energy_report"
)

print("Final report saved successfully!")


# Inspect the generated Parquet part files

import os

files = os.listdir("/content/final_energy_report")

for file in files:
    if file.startswith("part-"):
        print(file)